# SETReliable — Full-Cohort Colab Runner

This notebook runs the canonical SCMD pipeline from a verified code commit. It records runtime/input provenance, executes the full cohort without sampling, audits every saved held-out test observation, creates `colab_run_report.md`, and archives both `output/` and `data/processed/`.

**Important:** the full-cohort run is not executed by repository CI or by this notebook edit. Inventory, lead-time, scenario and stockout outcomes remain simulated.

In [ ]:
import os, platform, shutil, subprocess, sys, time, json, hashlib, zipfile
from pathlib import Path
from datetime import datetime, timezone

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def disk_snapshot(label, path='/content'):
    usage = shutil.disk_usage(path)
    row = {'stage': label, 'timestamp_utc': utc_now(), 'free_bytes': usage.free,
           'used_bytes': usage.used, 'total_bytes': usage.total}
    DISK_SNAPSHOTS.append(row)
    print(f"DISK[{label}] free={usage.free/1024**3:.2f} GB used={usage.used/1024**3:.2f} GB")
    return row

DISK_SNAPSHOTS = []
STAGE_RECORDS = []
RUN_STARTED_UTC = utc_now()
RUN_START_MONOTONIC = time.monotonic()
print('Python:', sys.version)
print('Platform:', platform.platform())
print('CPU cores:', os.cpu_count())
try:
    import psutil
    MEMORY_TOTAL_BYTES = int(psutil.virtual_memory().total)
    print('Memory total (GB):', round(MEMORY_TOTAL_BYTES / 1024**3, 2))
except ImportError:
    MEMORY_TOTAL_BYTES = None
    print('Memory total: unavailable (psutil not installed)')
disk_snapshot('initial')
if sys.version_info[:2] != (3, 11):
    raise RuntimeError(f'Python 3.11 is required; found {sys.version_info.major}.{sys.version_info.minor}')

In [ ]:
REPO_URL = 'https://github.com/Souptik-Hazra/SET_new.git'
REPO_COMMIT = '7dd7ca0ee291e5b34361faa9b52c89b80b773e08'
REPO_DIR = Path('/content/SET_new')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--prune', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REPO_COMMIT], check=True)
ACTUAL_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print('PINNED_COMMIT:', REPO_COMMIT)
print('CHECKED_OUT_COMMIT:', ACTUAL_COMMIT)
if ACTUAL_COMMIT != REPO_COMMIT:
    raise RuntimeError('Repository SHA mismatch; refusing to install dependencies or run any pipeline stage.')
disk_snapshot('after_checkout')

In [ ]:
def run_stage(name, command, cwd=None, log_path=None):
    started = time.monotonic()
    started_utc = utc_now()
    print(f'\n===== START {name} {started_utc} =====')
    log_file = Path(log_path) if log_path else None
    if log_file:
        log_file.parent.mkdir(parents=True, exist_ok=True)
        stream = log_file.open('a', encoding='utf-8')
    else:
        stream = None
    process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='')
            if stream:
                stream.write(line)
                stream.flush()
        return_code = process.wait()
    finally:
        if stream:
            stream.close()
    record = {'stage': name, 'command': list(map(str, command)),
              'started_utc': started_utc, 'ended_utc': utc_now(),
              'elapsed_seconds': round(time.monotonic()-started, 3),
              'exit_code': int(return_code)}
    STAGE_RECORDS.append(record)
    disk_snapshot(f'after_{name}')
    print(f'===== END {name} exit={return_code} elapsed={record["elapsed_seconds"]}s =====')
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
    return record

run_stage('install_pip', [sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'])
run_stage('install_requirements', [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], cwd=REPO_DIR)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATASET_DIR = Path('/content/drive/MyDrive/Dataset')
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f'Dataset folder not found: {DATASET_DIR}')
csv_files = sorted(p for p in DATASET_DIR.rglob('*') if p.is_file() and p.suffix.lower() == '.csv')
if len(csv_files) != 1:
    found = [str(p.relative_to(DATASET_DIR)) for p in csv_files]
    raise RuntimeError(
        f'Expected exactly one CSV anywhere inside {DATASET_DIR}; found {len(csv_files)}. '
        f'Candidates: {found}. Keep one source CSV or change DATASET_DIR explicitly.'
    )
SOURCE_CSV = csv_files[0]
BOOTSTRAP_RESAMPLES = 2000
source_bytes = SOURCE_CSV.stat().st_size
SOURCE_SHA256 = hashlib.sha256()
with SOURCE_CSV.open('rb') as source_stream:
    for chunk in iter(lambda: source_stream.read(8 * 1024 * 1024), b''):
        SOURCE_SHA256.update(chunk)
SOURCE_SHA256 = SOURCE_SHA256.hexdigest()
import duckdb
con = duckdb.connect()
try:
    source_rows = int(con.execute(
        'SELECT COUNT(*) FROM read_csv_auto(?, header=true)', [str(SOURCE_CSV)]
    ).fetchone()[0])
finally:
    con.close()
if source_rows < 1:
    raise RuntimeError('Source CSV contains no data rows.')
free_bytes = shutil.disk_usage('/content').free
MIN_FREE_BYTES = 10 * 1024**3
print('DATASET_DIR:', DATASET_DIR)
print('SOURCE_CSV:', SOURCE_CSV)
print('SOURCE_BYTES:', source_bytes)
print('SOURCE_SHA256:', SOURCE_SHA256)
print('SOURCE_DATA_ROWS:', source_rows)
print('DISK_PREFLIGHT_MINIMUM_GB:', round(MIN_FREE_BYTES / 1024**3, 2))
print('DISK_FREE_GB:', round(free_bytes / 1024**3, 2))
if free_bytes < MIN_FREE_BYTES:
    raise RuntimeError('Insufficient free disk: at least 10 GB is required before starting the full-cohort pipeline.')
disk_snapshot('after_input_validation')

## Clean full-cohort execution

Only generated `data/processed/` and `output/` directories are removed. The source CSV is never modified. The canonical runner is invoked once with the complete input; no sampling, cohort truncation or scenario skipping is performed.

In [ ]:
for relative in ('data/processed', 'output'):
    target = REPO_DIR / relative
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)
PIPELINE_LOG = REPO_DIR / 'output' / 'colab_pipeline.log'
PIPELINE_STATUS = 'NOT_STARTED'
PIPELINE_ERROR = None
PIPELINE_STARTED_UTC = utc_now()
pipeline_command = [sys.executable, 'scripts/run_pipeline.py', '--source', str(SOURCE_CSV),
                    '--bootstrap', str(BOOTSTRAP_RESAMPLES)]
try:
    run_stage('canonical_pipeline', pipeline_command, cwd=REPO_DIR, log_path=PIPELINE_LOG)
    PIPELINE_STATUS = 'PASS'
except Exception as exc:
    PIPELINE_STATUS = 'FAIL'
    PIPELINE_ERROR = f'{type(exc).__name__}: {exc}'
    print('PIPELINE_FAILED:', PIPELINE_ERROR)
PIPELINE_ENDED_UTC = utc_now()

In [ ]:
import json
AUDIT_STATUS = 'NOT_RUN'
AUDIT = None
if PIPELINE_STATUS == 'PASS':
    try:
        run_stage('independent_empirical_audit',
                  [sys.executable, '-m', 'src.set_forecasting.validation.empirical_monotonicity'],
                  cwd=REPO_DIR, log_path=PIPELINE_LOG)
        audit_path = REPO_DIR / 'output' / 'empirical_monotonicity_audit.json'
        AUDIT = json.loads(audit_path.read_text(encoding='utf-8'))
        AUDIT_STATUS = AUDIT.get('status', 'MISSING_STATUS')
        print('EMPIRICAL_AUDIT_STATUS:', AUDIT_STATUS)
        print('TOTAL_TEST_OBSERVATIONS:', AUDIT.get('total_test_observations'))
        print('DISTRIBUTION_PASS/FAIL:', AUDIT.get('distribution_pass'), AUDIT.get('distribution_fail'))
        print('RADCSR_PASS/FAIL:', AUDIT.get('radcsr_pass'), AUDIT.get('radcsr_fail'))
        print('LOW_AVAILABILITY_FAILURES:', AUDIT.get('low_availability_failures'))
        print('PER_SCENARIO:', json.dumps(AUDIT.get('per_scenario', {}), indent=2))
        if AUDIT_STATUS != 'PASS' or AUDIT.get('low_availability_failures', 0):
            raise RuntimeError('Empirical monotonicity gate failed.')
    except Exception as exc:
        AUDIT_STATUS = 'FAIL'
        AUDIT_ERROR = f'{type(exc).__name__}: {exc}'
        print('EMPIRICAL_AUDIT_FAILED:', AUDIT_ERROR)
else:
    AUDIT_STATUS = 'SKIPPED_PIPELINE_FAILURE'
    AUDIT_ERROR = 'Pipeline did not complete successfully.'

In [ ]:
import csv, json, math
import pyarrow.parquet as pq

def read_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding='utf-8')) if path.is_file() else None

def read_csv_rows(path):
    path = Path(path)
    if not path.is_file():
        return None
    with path.open('r', encoding='utf-8-sig', newline='') as stream:
        return list(csv.DictReader(stream))

def artifact_inventory(root):
    result = []
    for path in sorted(p for p in root.rglob('*') if p.is_file()):
        item = {'path': str(path.relative_to(REPO_DIR)), 'bytes': path.stat().st_size}
        if path.suffix.lower() == '.parquet':
            try:
                item['rows'] = int(pq.ParquetFile(path).metadata.num_rows)
            except Exception as exc:
                item['parquet_metadata_error'] = str(exc)
        result.append(item)
    return result

OUT = REPO_DIR / 'output'
PROC = REPO_DIR / 'data' / 'processed'
pipeline_report = read_json(OUT / 'pipeline_report.json') or {}
phase1 = read_json(OUT / 'phase1_diagnostics.json') or {}
audit = read_json(OUT / 'empirical_monotonicity_audit.json') or AUDIT or {}
cohort_rows = read_csv_rows(OUT / 'tables' / 'cohort_inclusion_report.csv')
monthly_rows = read_csv_rows(OUT / 'tables' / 'monthly_data_summary.csv')
cohort_summary = {'status': 'ABSENT'}
if cohort_rows is not None:
    included = [r for r in cohort_rows if str(r.get('COHORT_INCLUDED', '')).lower() == 'true']
    excluded = [r for r in cohort_rows if str(r.get('COHORT_INCLUDED', '')).lower() == 'false']
    reasons = {}
    for row in excluded:
        for reason in filter(None, str(row.get('EXCLUSION_REASON', '')).split(';')):
            reasons[reason] = reasons.get(reason, 0) + 1
    cohort_summary = {'series_before_filter': len(cohort_rows), 'series_included': len(included),
                      'series_excluded': len(excluded), 'exclusion_reason_counts': reasons}

monthly_gap_report = {'status': 'ABSENT'}
if monthly_rows is not None:
    months = sorted({str(r.get('YEAR_MONTH', '')) for r in monthly_rows if r.get('YEAR_MONTH')})
    parsed_months = []
    for value in months:
        try:
            parsed_months.append((int(value[:4]), int(value[4:6])))
        except Exception:
            pass
    missing_months = []
    if parsed_months:
        cursor = parsed_months[0]
        end = parsed_months[-1]
        present = set(parsed_months)
        while cursor <= end:
            if cursor not in present:
                missing_months.append(f'{cursor[0]:04d}{cursor[1]:02d}')
            cursor = (cursor[0] + (cursor[1] == 12), 1 if cursor[1] == 12 else cursor[1] + 1)
    monthly_gap_report = {'months_present': len(parsed_months), 'first_month': months[0] if months else None,
                          'last_month': months[-1] if months else None, 'missing_calendar_months': missing_months}

table_summaries = {}
for name in ['forecast_metrics.csv', 'risk_brier_by_scenario.csv', 'risk_calibration_metrics.csv',
             'phase7_ablation_metrics.csv', 'radcsr_vs_baselines_bootstrap_holm.csv',
             'phase6_inventory_scenarios.csv', 'reliability_baseline_metrics.csv']:
    rows = read_csv_rows(OUT / 'tables' / name)
    table_summaries[name] = {'status': 'ABSENT'} if rows is None else {'rows': len(rows), 'columns': list(rows[0]) if rows else []}

required_paths = [
    'output/pipeline_report.json', 'output/phase1_diagnostics.json', 'output/pipeline_stage_log.json', 'output/final_verdict.md',
    'output/empirical_monotonicity_audit.json', 'output/radcsr_monotonicity_audit.json',
    'output/tables/cohort_inclusion_report.csv', 'output/tables/forecast_metrics.csv',
    'output/tables/risk_brier_by_scenario.csv', 'output/tables/risk_calibration_metrics.csv',
    'output/tables/phase7_ablation_metrics.csv', 'output/tables/radcsr_vs_baselines_bootstrap_holm.csv',
    'data/processed/scmd_monthly_series.parquet', 'data/processed/forecasts.parquet',
    'data/processed/scenario_streams.parquet', 'data/processed/simulated_inventory.parquet',
    'data/processed/radcsr_risk_details.parquet', 'data/processed/calibrated_risk_probabilities.parquet',
]
artifact_checklist = []
for relative in required_paths:
    path = REPO_DIR / relative
    item = {'path': relative, 'status': 'PRESENT' if path.is_file() and path.stat().st_size else 'MISSING'}
    if item['status'] == 'PRESENT':
        item['bytes'] = path.stat().st_size
        if path.suffix == '.parquet':
            item['rows'] = int(pq.ParquetFile(path).metadata.num_rows)
    artifact_checklist.append(item)

forecast_info = pipeline_report.get('forecast', {}) if isinstance(pipeline_report.get('forecast', {}), dict) else {'raw': pipeline_report.get('forecast')}
report_data = {
    'run_started_utc': RUN_STARTED_UTC, 'run_ended_utc': utc_now(),
    'elapsed_seconds': round(time.monotonic() - RUN_START_MONOTONIC, 3),
    'repository_url': REPO_URL, 'pinned_commit': REPO_COMMIT, 'checked_out_commit': ACTUAL_COMMIT,
    'python': sys.version, 'platform': platform.platform(), 'cpu_cores': os.cpu_count(),
    'memory_total_bytes': MEMORY_TOTAL_BYTES, 'source_csv': str(SOURCE_CSV),
    'source_bytes': source_bytes, 'source_sha256': SOURCE_SHA256, 'source_data_rows': source_rows,
    'bootstrap_resamples_requested': BOOTSTRAP_RESAMPLES,
    'pipeline_status': PIPELINE_STATUS, 'pipeline_error': PIPELINE_ERROR,
    'audit_status': AUDIT_STATUS, 'audit_error': globals().get('AUDIT_ERROR'),
    'stage_records': STAGE_RECORDS, 'pipeline_stage_log': read_json(OUT / 'pipeline_stage_log.json'), 'disk_snapshots': DISK_SNAPSHOTS,
    'cohort': cohort_summary, 'phase1_diagnostics': phase1,
    'monthly_calendar_coverage': monthly_gap_report,
    'forecast_provenance_and_fallback': forecast_info,
    'pipeline_metrics': pipeline_report,
    'brier_calibration_ablation_bootstrap_holm_tables': table_summaries,
    'scenario_evidence': {'scenario_rows': pipeline_report.get('scenario_rows'),
                          'mean_unreliability_by_test_scenario': pipeline_report.get('mean_unreliability_by_test_scenario'),
                          'mean_radcsr_probability_by_test_scenario': pipeline_report.get('mean_radcsr_probability_by_test_scenario'),
                          'scenario_reliability_acceptance': pipeline_report.get('scenario_reliability_acceptance')},
    'empirical_monotonicity': audit, 'artifact_checklist': artifact_checklist,
    'all_output_artifacts': artifact_inventory(OUT), 'all_processed_artifacts': artifact_inventory(PROC),
    'limitations': 'SCMD contains no observed inventory, receipts, supplier lead times or stockout events. Inventory, scenarios, lead-time demand and stockout labels are simulated. Simulated Brier/calibration metrics are not evidence of observed stockout prediction.',
    'interpretation': 'Evidence only; no research claim verdict is issued by this notebook.'
}
report_path = OUT / 'colab_run_report.md'
report_path.parent.mkdir(parents=True, exist_ok=True)
report_text = '# Colab Full-Cohort Run Report\n\n'
report_text += '```json\n' + json.dumps(report_data, indent=2, default=str) + '\n```\n\n'
report_text += 'Empirical evidence collected. Claim evaluation pending review.\n'
report_path.write_text(report_text, encoding='utf-8')
print('COLAB_REPORT:', report_path)
print('REQUIRED_ARTIFACTS_MISSING:', [x['path'] for x in artifact_checklist if x['status'] != 'PRESENT'])

In [ ]:
archive = Path('/content/SET_new_full_run.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for relative_root in ('output', 'data/processed'):
        root = REPO_DIR / relative_root
        if root.exists():
            for file in sorted(p for p in root.rglob('*') if p.is_file()):
                zf.write(file, arcname=str(file.relative_to(REPO_DIR)))
drive_target = Path('/content/drive/MyDrive/SET_new_full_run.zip')
shutil.copy2(archive, drive_target)
report_drive = Path('/content/drive/MyDrive/colab_run_report.md')
shutil.copy2(report_path, report_drive)
print('FULL_RUN_ARCHIVE:', drive_target)
print('COLAB_RUN_REPORT:', report_drive)
print('PIPELINE_STATUS:', PIPELINE_STATUS, 'AUDIT_STATUS:', AUDIT_STATUS)
if PIPELINE_STATUS != 'PASS' or AUDIT_STATUS != 'PASS':
    raise RuntimeError('Full-cohort run or empirical monotonicity gate failed. Review colab_run_report.md and colab_pipeline.log; do not interpret results as accepted.')
if any(x['status'] != 'PRESENT' for x in artifact_checklist):
    raise RuntimeError('Required artifact checklist is incomplete; see colab_run_report.md.')